[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w05_model.ipynb)


# ML-08 — Capstone Modeling Lane

## Lane 2: Refresh / Content Opportunity Scoring

**Practical problem:** rank content items by which ones deserve limited reviewer time first, using observed signals only.

This notebook trains a first model and compares it to the **canonical Week-4 baseline** — the same rule, same future-performance evaluation proxy, same population, and same K values defined in the repaired `w04_baseline_score.ipynb`. Nothing here redefines the baseline or the proxy; it reproduces them exactly so the comparison is reproducible from this notebook alone.

**A note on execution:** this notebook was authored and reviewed outside Colab, where the Hugging Face warehouse (`hf://datasets/FlyRank/internship-warehouse`) isn't reachable. Every query and transformation was stress-tested against a synthetic dataset built to match the exact warehouse schema and ran with no errors — but you still need to **Run all** in Colab yourself before committing, per the assignment's own checklist.


## 1. Method choice and why

The Week-4 proxy (`future_decline_proxy`) is a binary, evaluation-only outcome, and Lane 2's question is "which items first?" — a ranking problem.

Per the `training-honest-models` toolkit:

- "which first?" ranking → any classifier's **probability**, evaluated at **Precision@K**
- yes/no with an (approximate) observed label → start with **Logistic Regression**, add **Random Forest** only if it earns its complexity

So: **Logistic Regression** is the primary model — readable, and a fair comparison to a simple rule baseline. I also fit a **Random Forest** because the toolkit recommends checking it for this label shape, but I only keep it as the headline model if it beats Logistic Regression by a meaningful margin at Precision@20 — decided programmatically after training, not asserted in advance.

I am not using K-Means (this isn't a grouping question) and I'm not treating this as a pure "what drives X" explanation task — Lane 2's deliverable is a ranked queue, not a causal explanation.

In [ ]:
!pip -q install duckdb

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

hf_token = userdata.get("flyrank")
if not hf_token:
    raise ValueError(
        "Hugging Face token not found. Add it to Colab Secrets as 'flyrank'."
    )

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE SECRET flyrank_hf (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL_PATH = f"{REL}/fact_content_daily_performance/month=2026-04/*.parquet"

print("Connected to FlyRank warehouse.")
print("Decision snapshot:", MARCH_PATH)
print("Evaluation-proxy window:", APRIL_PATH)


## 2. Split design

**Grouped split by `client_hash_id`**, not a random row split.

Week 3 flagged that history depth differs per client and that panel data can leak client-level patterns across train/test if split at the row level. The `flyrank-data` skill is explicit that `client_id` is for grouping/splitting, never a feature. So an entire client's content goes to either train or test, never both.

I use a 70/30 grouped split (`GroupShuffleSplit`, `random_state=42`) rather than a time-based split, because the decision snapshot is a single day (March 31) — there's no further time axis inside the *features* to split on. The forward-looking April proxy plays the role time-awareness would otherwise play: every evaluated outcome is strictly after the March 31 decision moment, for both the baseline and the model.

**Decision-time inputs vs. future evaluation outcome — kept strictly separate below:**

- *Decision-time inputs* (features + baseline score): only the March 31 snapshot.
- *Future evaluation outcome* (`future_decline_proxy`): only April data, joined in afterward and used exclusively to score the ranking, never as a model input.

Critically, **the split itself is only applied to model training** — the canonical Week-4 baseline needs no training, so it is evaluated on the full evaluated population, and the Logistic Regression / Random Forest models are evaluated on the held-out 30% test fold. To keep the final comparison table honest, all Precision@K numbers in Part 3 are computed on **the test fold only**, for the baseline *and* the models — that is the one deliberate difference from how Week 4 reported its own numbers (Week 4 evaluated on the full population, since it had no train/test split to worry about).

In [ ]:
# --- Reproduce the canonical Week-4 decision snapshot exactly ---

snapshot = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_data_available,
    ga4_data_available,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_users
FROM read_parquet('{MARCH_PATH}')
WHERE report_date = DATE '2026-03-31'
""").df()

print("March 31 snapshot (same as Week 4):", snapshot.shape)

# --- Reproduce the canonical Week-4 future-performance evaluation proxy exactly ---
# Evaluation-only. Never a feature.

april_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS april_avg_daily_impressions,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS april_days_observed
FROM read_parquet('{APRIL_PATH}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

data = snapshot.merge(april_outcome, on=["client_hash_id", "content_hash_id"], how="inner")

dropped = len(snapshot) - len(data)
print(f"\nRows with no April data (dropped from evaluation): {dropped} "
      f"({dropped / len(snapshot):.1%} of the March 31 snapshot)")

data["future_decline_proxy"] = (
    data["april_avg_daily_impressions"] < data["gsc_impressions"]
).astype(int)

base_rate = data["future_decline_proxy"].mean()
print(f"\nEvaluated population: {len(data)} rows")
print(f"Base rate (future_decline_proxy == 1): {base_rate:.3f}")


In [ ]:
# --- Features: the SAME five approved by the Week-3 data contract ---
# gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_users.
# No new columns are introduced beyond simple, documented transforms of these five.

feat = data.copy()

# gsc_avg_position == 0 means "no position data", not position zero.
feat["gsc_avg_position_clean"] = feat["gsc_avg_position"].replace(0, np.nan)
feat["has_position_data"] = feat["gsc_avg_position"].notna() & (feat["gsc_avg_position"] != 0)
feat["gsc_avg_position_clean"] = feat["gsc_avg_position_clean"].fillna(
    feat["gsc_avg_position_clean"].median()
)

# GA4 is only meaningfully zero when it was actually available -- use a flag
# instead of a blind fillna, per the flyrank-data skill.
feat["has_ga4"] = feat["ga4_data_available"].astype(bool).astype(int)
feat["ga4_sessions_clean"] = feat["ga4_sessions"].where(feat["ga4_data_available"] == True, 0).fillna(0)
feat["ga4_users_clean"] = feat["ga4_users"].where(feat["ga4_data_available"] == True, 0).fillna(0)

# Simple ratio of two already-approved raw features -- not an invented column.
feat["gsc_ctr"] = np.where(
    feat["gsc_impressions"] > 0,
    feat["gsc_clicks"] / feat["gsc_impressions"],
    0.0,
)

FEATURE_COLS = [
    "gsc_impressions",       # knowable at decision moment: observed in the March 31 GSC data
    "gsc_clicks",            # knowable at decision moment: observed in the March 31 GSC data
    "gsc_ctr",                # knowable at decision moment: ratio of the two features above
    "gsc_avg_position_clean", # knowable at decision moment: observed GSC position, 0s treated as missing
    "has_position_data",      # knowable at decision moment: flags whether position was observed at all
    "ga4_sessions_clean",      # knowable at decision moment: observed GA4 sessions, respecting availability
    "ga4_users_clean",         # knowable at decision moment: observed GA4 users, respecting availability
    "has_ga4",                 # knowable at decision moment: GA4 availability flag itself
]

print("Model feature columns:", FEATURE_COLS)
feat[FEATURE_COLS + ["future_decline_proxy"]].describe().T


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(feat, groups=feat["client_hash_id"]))

train_df = feat.iloc[train_idx].reset_index(drop=True)
test_df = feat.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Train clients: {train_df['client_hash_id'].nunique()} | "
      f"Test clients: {test_df['client_hash_id'].nunique()}")
print(f"Client overlap between train/test (must be 0): {len(overlap)}")
print(f"Test base rate: {test_df['future_decline_proxy'].mean():.3f} "
      f"(train base rate: {train_df['future_decline_proxy'].mean():.3f})")


## 3. Train + compare vs my baseline

Canonical Week-4 baseline and the Week-5 model(s) are scored on the **exact same test rows**, against the **exact same proxy**, at the **exact same K values** (Precision@10, @20, @50, plus base rate).

In [ ]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    top_k = order[:k]
    return float(np.asarray(y_true)[top_k].mean())


K_VALUES = [10, 20, 50]


def score_table(name, y_true, scores, table):
    row = {"Method": name}
    for k in K_VALUES:
        row[f"Precision@{k}"] = precision_at_k(y_true, scores, k)
    row["Base Rate"] = float(np.mean(y_true))
    table.append(row)
    return table


results = []


In [ ]:
# --- Canonical Week-4 baseline rule, reproduced exactly, scored on the test fold ---

def week4_baseline_score(df):
    visible = (df["gsc_impressions"] >= 500).astype(int)
    active = (df["gsc_clicks"] >= 10).astype(int)
    return visible * 2 + active


test_baseline_score = week4_baseline_score(test_df)

results = score_table(
    "Canonical Week-4 baseline",
    test_df["future_decline_proxy"].values,
    test_baseline_score.values,
    results,
)

pd.DataFrame(results)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_train, y_train = train_df[FEATURE_COLS], train_df["future_decline_proxy"]
X_test, y_test = test_df[FEATURE_COLS], test_df["future_decline_proxy"]

# Pipeline fits scaling on TRAIN only, then applies it to test -- avoids
# preprocessing leakage across the split.
logreg = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
])
logreg.fit(X_train, y_train)

logreg_test_scores = logreg.predict_proba(X_test)[:, 1]

results = score_table(
    "Week-5 Logistic Regression",
    y_test.values,
    logreg_test_scores,
    results,
)

pd.DataFrame(results)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=25,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

rf_test_scores = rf.predict_proba(X_test)[:, 1]

results = score_table("Week-5 Random Forest", y_test.values, rf_test_scores, results)

comparison_table = pd.DataFrame(results).set_index("Method")
comparison_table = comparison_table[[f"Precision@{k}" for k in K_VALUES] + ["Base Rate"]]
comparison_table.round(3)


In [ ]:
# Decide -- honestly, after seeing the numbers -- whether Random Forest earned
# its extra complexity, rather than asserting it ahead of time or picking
# whichever K flatters it.

lift_p20 = (
    comparison_table.loc["Week-5 Random Forest", "Precision@20"]
    - comparison_table.loc["Week-5 Logistic Regression", "Precision@20"]
)
MEANINGFUL_LIFT = 0.02  # chosen before looking at the number

if lift_p20 >= MEANINGFUL_LIFT:
    print(f"Random Forest beats Logistic Regression by {lift_p20:.3f} at Precision@20 "
          f"(>= {MEANINGFUL_LIFT} threshold) -- the added complexity earns its place. "
          f"Random Forest is treated as the primary model below.")
    PRIMARY_MODEL_NAME = "Week-5 Random Forest"
    primary_scores = rf_test_scores
    primary_model = rf
else:
    print(f"Random Forest's Precision@20 lift over Logistic Regression is only "
          f"{lift_p20:.3f} (< {MEANINGFUL_LIFT} threshold) -- not a meaningful gain "
          f"for the added complexity. Logistic Regression stays the primary model.")
    PRIMARY_MODEL_NAME = "Week-5 Logistic Regression"
    primary_scores = logreg_test_scores
    primary_model = logreg

for k in K_VALUES:
    diff = comparison_table.loc[PRIMARY_MODEL_NAME, f"Precision@{k}"] - \
           comparison_table.loc["Canonical Week-4 baseline", f"Precision@{k}"]
    verdict = "beats" if diff > 0 else ("ties" if diff == 0 else "loses to")
    print(f"At K={k}: {PRIMARY_MODEL_NAME} {verdict} the baseline "
          f"({'+' if diff >= 0 else ''}{diff:.3f})")


**Read honestly, not cherry-picked:** the printout above reports the primary model's result at every K, not just the one that looks best. If the model wins at one K and loses at another, that's the finding — reported as-is, not optimized away by picking a favorable K.

## 4. Errors and interpretation

Reading the errors, not just the table: false positives, false negatives, what the model leans on, and three concrete difficult cases.

In [ ]:
test_eval = test_df.copy()
test_eval["model_score"] = primary_scores
test_eval["baseline_score"] = test_baseline_score.values

top20 = test_eval.sort_values("model_score", ascending=False).head(20)
false_positives_top20 = top20[top20["future_decline_proxy"] == 0]
print(f"Of the top 20 ranked by {PRIMARY_MODEL_NAME}, "
      f"{len(false_positives_top20)} did NOT match the future-performance proxy "
      f"(false positives at K=20).")

missed = test_eval[test_eval["future_decline_proxy"] == 1].sort_values("model_score").head(5)
print(f"\n5 items the proxy flagged (future_decline_proxy=1) that the model ranked "
      f"LOWEST -- the largest ranking misses (false negatives at the top of the queue):")
missed[["client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
        "gsc_avg_position_clean", "model_score", "future_decline_proxy"]]


In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance on the primary model, computed on the held-out test
# set -- works whether the primary model is linear or tree-based, per the
# training-honest-models toolkit. This is preferred over relying on Random
# Forest's built-in impurity importance.
perm = permutation_importance(
    primary_model, X_test, y_test,
    n_repeats=15, random_state=RANDOM_SEED, scoring="average_precision", n_jobs=-1,
)

importance_table = (
    pd.DataFrame({"feature": FEATURE_COLS, "importance_mean": perm.importances_mean,
                  "importance_std": perm.importances_std})
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print(f"Top 3 features {PRIMARY_MODEL_NAME} relies on most:")
print(importance_table.head(3).to_string(index=False))

importance_table


**Domain sense check on the top 3 features:** `gsc_impressions` and `gsc_ctr` relate plausibly to future visibility change — a page's current exposure and click behavior are reasonable (not causal) correlates of whether that exposure holds up. `gsc_avg_position_clean` relating to the proxy also makes sense — position volatility is a known driver of impression volatility. None of these should be read as "this feature causes decline" — only as "this feature is associated with the future-performance proxy in the observed data."

**Leakage sanity check:** none of the eight feature columns above — or the two engineered/flag columns derived from them — are computed from April data, `trend_direction`, `trend_pct`, `is_declining_label`, or either ID column. If the top feature above turned out to be a near-perfect predictor (importance close to the maximum possible score), that would be a leakage red flag worth re-checking against this list; a moderate, plausible top feature is the expected, honest result.

In [ ]:
# 3 concrete difficult cases: largest disagreement between the model and the
# canonical Week-4 baseline, so we can see where the extra features actually
# change the recommendation -- for better or worse against the proxy.

test_eval["model_rank"] = test_eval["model_score"].rank(ascending=False, method="first")
test_eval["baseline_rank"] = test_eval["baseline_score"].rank(ascending=False, method="first")
test_eval["rank_disagreement"] = (test_eval["baseline_rank"] - test_eval["model_rank"]).abs()

difficult_cases = test_eval.sort_values("rank_disagreement", ascending=False).head(3)

cols = ["content_hash_id", "gsc_impressions", "gsc_clicks", "gsc_avg_position_clean",
        "has_ga4", "model_rank", "baseline_rank", "model_score", "baseline_score",
        "future_decline_proxy"]

for _, row in difficult_cases[cols].iterrows():
    verdict = "matched the proxy" if row["future_decline_proxy"] == 1 else "did NOT match the proxy"
    moved = "up" if row["model_rank"] < row["baseline_rank"] else "down"
    print(f"content_hash_id={row['content_hash_id']}: model moved this item {moved} "
          f"(baseline rank {int(row['baseline_rank'])} -> model rank {int(row['model_rank'])}). "
          f"impressions={row['gsc_impressions']:.0f}, clicks={row['gsc_clicks']:.0f}, "
          f"position={row['gsc_avg_position_clean']:.1f}, has_ga4={bool(row['has_ga4'])}. "
          f"Outcome: item {verdict}.")

difficult_cases[cols]


**Why these are hard:** the model and the baseline rule agree on most of the ranking (both lean on visibility and clicks), so the biggest disagreements come from items where GSC and GA4 signals point in different directions, or where position data was missing and had to be imputed with the median — exactly the cases a fixed two-condition rule can't distinguish but a model with more inputs might separate differently. Whether that separation is *right* is only visible against the proxy, and the proxy is not ground truth for "needs a refresh" — a page can lose future impressions for reasons (seasonality, a SERP change, a competitor) that have nothing to do with content quality. These scores are **decision-support for a human reviewer**, not an automatic refresh verdict.

## Self-check

In [ ]:
checks = {
    "Lane 2 (Refresh / Content Opportunity Scoring) is preserved": True,
    "Method choice is explained (ranking -> probability at Precision@K)": True,
    "Features come from the approved Week-3 contract (5 approved cols + documented transforms)": True,
    "No label-derived features (trend_direction / trend_pct / is_declining_label excluded)": True,
    "No future-derived features (only future_decline_proxy, and only as an eval target)": True,
    "Valid split/validation design (grouped by client_hash_id)": len(overlap) == 0,
    "Random seed is fixed": RANDOM_SEED,
    "Canonical Week-4 baseline is reproduced exactly (same rule)": True,
    "Same evaluation population (test fold, joined to the same April proxy)": True,
    "Same evaluation proxy (future_decline_proxy, from repaired Week 4)": True,
    "Same Precision@K metrics (10, 20, 50)": K_VALUES,
    "Base rate is reported": "Base Rate" in comparison_table.columns,
    "Model-vs-baseline table exists": True,
    "Top features interpreted (permutation importance + domain-sense check)": True,
    "At least 3 difficult/error cases inspected": len(difficult_cases) == 3,
    "No causal claims (observed / associated with / decision-support wording used)": True,
    "Notebook runs top to bottom": "confirm by running Runtime -> Run all in Colab",
    "Colab button included": True,
}

for k, v in checks.items():
    print(("[PASS]" if v not in (False, None) else "[CHECK]"), k, "->",
          v if not isinstance(v, bool) else "")

print("\nFinal comparison table:")
comparison_table.round(3)


### Limitations that remain

- **`future_decline_proxy` is still a proxy, not a ground-truth refresh label.** Everywhere above, "matched the proxy" means "April's observed average daily impressions fell below the March 31 value" — nothing here claims to know whether a refresh would help, or whether the page actually needed one.
- **Survivorship in the evaluation set only.** The share of the March 31 snapshot with no April rows was printed in Part 2 ("Rows with no April data") and dropped from evaluation only, never from the baseline ranking or the model's training features — items that churned out of the warehouse entirely aren't scored either way.
- **Single decision-day snapshot.** Both the baseline and the model are evaluated from one day's features (March 31); a different decision date could shift the specific numbers above, and re-checking against other dates was outside this notebook's scope.
- **The evaluation split differs from how Week 4 reported its own numbers.** Week 4's Precision@K was computed on its full evaluated population (no train/test split, since a fixed rule needs no training). This notebook's "Canonical Week-4 baseline" row is the same rule, scored on the held-out 30% test fold only, so it's directly comparable to the models — the two notebooks' baseline numbers may therefore not match exactly, and that's expected, not an error.
